<a href="https://colab.research.google.com/github/anaghaayyagari/AAI2025/blob/main/Exercise3_Self_Reflection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 3 — Self-Reflection Prompt for Improving Output

**Goal:** ask the AI to critique its own summary against explicit requirements and then produce an improved version.

### Flow
1. **Original summary (Version A)**: a deliberately minimal prompt: *"Summarize this text."*
2. **Self-critique**: the model scores its summary 1–5 on six explicit criteria and quotes specific problems (JSON).
3. **Revise**: the model rewrites the summary to fix every issue, using only facts from the source.
4. Steps 2–3 repeat until every criterion scores ≥ 4 (max 3 rounds). Word count is also measured in code.
5. **Before/after comparison.**

## Setup
**Tools used:** Google Colab + OpenAI API (Python `openai` library). You can switch `PROVIDER` to `"anthropic"` to use the Claude API instead.

**API key:** open the 🔑 **Secrets** panel in Colab's left sidebar, add `OPENAI_API_KEY` (or `ANTHROPIC_API_KEY`), and turn on notebook access.
**Never paste your key into a cell.** This notebook will be public on GitHub.

In [ ]:
!pip install -q openai anthropic

import os, json, re, textwrap

PROVIDER = "openai"   # "openai" or "anthropic"
MODEL = "gpt-4o-mini" if PROVIDER == "openai" else "claude-sonnet-5"   # change if your account uses another model

KEY_NAME = "OPENAI_API_KEY" if PROVIDER == "openai" else "ANTHROPIC_API_KEY"
try:
    from google.colab import userdata
    os.environ[KEY_NAME] = userdata.get(KEY_NAME)   # read from Colab Secrets, never hard-coded
except ImportError:
    pass  # running outside Colab: set the environment variable yourself

if PROVIDER == "openai":
    from openai import OpenAI
    client = OpenAI()
else:
    import anthropic
    client = anthropic.Anthropic()


def banner(title, body):
    print(f"\n{'=' * 70}\n{title}\n{'-' * 70}")
    print(textwrap.indent(str(body), "  "))


def chat(system, user, label="LLM call", show=True):
    """Send one system + user prompt to the model and return the reply text."""
    if PROVIDER == "openai":
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        )
        text = resp.choices[0].message.content or ""
    else:
        resp = client.messages.create(
            model=MODEL, max_tokens=1500, system=system,
            messages=[{"role": "user", "content": user}],
        )
        text = "".join(b.text for b in resp.content if b.type == "text")
    text = text.strip()
    if show:
        banner(label, text)
    return text


def extract_json(text):
    """Pull the first JSON object out of a model reply (tolerates ```json fences)."""
    cleaned = re.sub(r"```(?:json)?", "", text)
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start == -1 or end == -1:
        raise ValueError(f"No JSON object found in model output:\n{text}")
    return json.loads(cleaned[start:end + 1])


print(f"Ready: provider={PROVIDER}, model={MODEL}")

## Source text

In [ ]:
SOURCE_TEXT = """Northwind Logistics ran a six-month pilot of a four-day, 32-hour work week in its Customer Operations and Fulfillment departments, covering 120 employees. Pay was unchanged. The goal was to test whether the company could keep output steady while reducing burnout, which had driven voluntary turnover to 18% the previous year.

Results were mostly positive. Shipments processed per employee rose 3% compared with the same period last year, and order error rates were flat. Sick days fell 22%, and in the end-of-pilot survey 87% of participants said they wanted to keep the schedule. Two employees who had given notice before the pilot withdrew their resignations.

The pilot was not without costs. In the first month, average customer response times slowed by 8% as teams adjusted to staggered days off. Response times returned to normal by the third month after the company introduced a shared coverage calendar. Some managers reported spending more time on scheduling, and two team leads said the compressed week made cross-department meetings harder to arrange.

After reviewing the results, the leadership team decided not to roll the schedule out company-wide yet. Instead, it will extend the pilot to the Returns and Billing departments starting in Q1, on the condition that customer response times stay within the current service target. A decision on a company-wide policy is expected after another six months of data."""

## Requirements used for the critique
| Requirement | Value |
|---|---|
| Length | 60 words or fewer |
| Audience | busy executives who need the result and the decision fast |
| Format | one plain paragraph, no bullet points or headings |
| Focus | main result with key numbers, the main trade-off, the final decision |
| Accuracy | no claims or numbers that aren't in the source |
| Pass bar | every criterion ≥ 4/5, max 3 rounds |

## Prompts used

**Original summary prompt (Version A)** — system: `You are a helpful assistant.` · user: `Summarize this text: {source}`

**Self-critique (system prompt)**

```text
You are a strict editor reviewing a summary against its source text.
Score each criterion from 1 (poor) to 5 (excellent):
  coverage      - includes the main result with key numbers, the main trade-off, and the final decision
  accuracy      - every claim and number is supported by the source; nothing invented or exaggerated
  concision     - 60 words or fewer, no filler or vague phrases
  clarity       - a busy reader understands it in one pass
  audience_fit  - suited to busy executives who need the result and the decision fast
  format        - one plain paragraph, no bullet points or headings
Return ONLY a JSON object:
  "scores": {"coverage": n, "accuracy": n, "concision": n, "clarity": n, "audience_fit": n, "format": n}
  "issues": list of specific problems, quoting the words that cause each one
  "top_fix": the single most important change
Give a brief quality check, not a step-by-step reasoning transcript.
```

**Self-critique (user prompt)**

```text
SOURCE TEXT:
{source}

SUMMARY TO REVIEW:
{summary}
```

**Revision (system prompt)**

```text
You revise summaries using an editor's critique.
Fix every issue in the critique and keep what already works.
Use only facts from the source text - never add new information.
The summary is for busy executives who need the result and the decision fast, must be 60 words or fewer, and must be one plain paragraph, no bullet points or headings.
Before answering, check the revised summary against every issue in the critique.
Output only the revised summary, with no preamble.
```

**Revision (user prompt)**

```text
SOURCE TEXT:
{source}

CURRENT SUMMARY:
{summary}

EDITOR'S CRITIQUE:
{critique}
```

In [ ]:
# Requirements the summary is judged against
MAX_WORDS = 60
AUDIENCE = "busy executives who need the result and the decision fast"
REQUIRED_FORMAT = "one plain paragraph, no bullet points or headings"
PASS_SCORE = 4          # every criterion must score at least this
MAX_ROUNDS = 3          # critique -> revise rounds
CRITERIA = ["coverage", "accuracy", "concision", "clarity", "audience_fit", "format"]

# Version A: the original, deliberately minimal summary prompt
SUMMARIZE_SYSTEM = "You are a helpful assistant."
SUMMARIZE_USER = "Summarize this text:\n\n{source}"

# Self-critique prompt
CRITIQUE_SYSTEM = f"""You are a strict editor reviewing a summary against its source text.
Score each criterion from 1 (poor) to 5 (excellent):
  coverage      - includes the main result with key numbers, the main trade-off, and the final decision
  accuracy      - every claim and number is supported by the source; nothing invented or exaggerated
  concision     - {MAX_WORDS} words or fewer, no filler or vague phrases
  clarity       - a busy reader understands it in one pass
  audience_fit  - suited to {AUDIENCE}
  format        - {REQUIRED_FORMAT}
Return ONLY a JSON object:
  "scores": {{"coverage": n, "accuracy": n, "concision": n, "clarity": n, "audience_fit": n, "format": n}}
  "issues": list of specific problems, quoting the words that cause each one
  "top_fix": the single most important change
Give a brief quality check, not a step-by-step reasoning transcript."""

CRITIQUE_USER = "SOURCE TEXT:\n{source}\n\nSUMMARY TO REVIEW:\n{summary}"

# Revision prompt: requires a revised version that fixes every issue
REVISE_SYSTEM = f"""You revise summaries using an editor's critique.
Fix every issue in the critique and keep what already works.
Use only facts from the source text - never add new information.
The summary is for {AUDIENCE}, must be {MAX_WORDS} words or fewer, and must be {REQUIRED_FORMAT}.
Before answering, check the revised summary against every issue in the critique.
Output only the revised summary, with no preamble."""

REVISE_USER = "SOURCE TEXT:\n{source}\n\nCURRENT SUMMARY:\n{summary}\n\nEDITOR'S CRITIQUE:\n{critique}"

In [ ]:
def critique(summary, round_no):
    result = extract_json(chat(CRITIQUE_SYSTEM,
                               CRITIQUE_USER.format(source=SOURCE_TEXT, summary=summary),
                               label=f"SELF-CRITIQUE (round {round_no})"))
    # Measure length in code as well: LLM judges often miss word limits
    words = len(summary.split())
    result["word_count"] = words
    if words > MAX_WORDS:
        result["scores"]["concision"] = min(result["scores"]["concision"], 2)
        result["issues"].append(f"Measured length is {words} words; the limit is {MAX_WORDS}.")
    print(f"\n[Word count check] {words} words (limit {MAX_WORDS})  Scores: {json.dumps(result['scores'])}")
    return result


def passes(result):
    return all(result["scores"].get(c, 0) >= PASS_SCORE for c in CRITERIA)

## Step 1 — Original summary

In [ ]:
original_summary = chat(SUMMARIZE_SYSTEM, SUMMARIZE_USER.format(source=SOURCE_TEXT),
                        label="ORIGINAL SUMMARY (Version A)")

## Steps 2–3 — Self-critique and revise

In [ ]:
summary = original_summary
first_review = review = critique(summary, 1)
round_no = 1

while not passes(review) and round_no < MAX_ROUNDS:
    round_no += 1
    summary = chat(REVISE_SYSTEM,
                   REVISE_USER.format(source=SOURCE_TEXT, summary=summary,
                                      critique=json.dumps(review, indent=2)),
                   label=f"REVISED SUMMARY (round {round_no})")
    review = critique(summary, round_no)

improved_summary = summary
print(f"\nReflection loop {'passed the rubric' if passes(review) else f'stopped after {MAX_ROUNDS} rounds'}"
      f" after {round_no} round(s).")

## Before / after comparison

In [ ]:
print("BEFORE (original summary):\n" + original_summary)
print("\nAFTER (improved summary):\n" + improved_summary)

print(f"\n{'Criterion':16}{'Before':>10}{'After':>10}")
for c in CRITERIA:
    print(f"{c:16}{first_review['scores'].get(c, '-'):>10}{review['scores'].get(c, '-'):>10}")
total_before = sum(first_review["scores"].get(c, 0) for c in CRITERIA)
total_after = sum(review["scores"].get(c, 0) for c in CRITERIA)
print(f"{'TOTAL /' + str(5 * len(CRITERIA)):16}{total_before:>10}{total_after:>10}")
print(f"{'word count':16}{first_review['word_count']:>10}{review['word_count']:>10}")

print("\nIssues found in the original:", *first_review["issues"], sep="\n  - ")

### Notes
*(Fill in after running. What were the biggest differences between before and after? For example: a factual error fixed, key numbers added, length cut to fit the limit. Which requirement in the critique prompt drove the biggest improvement?)*